In [1]:
import itertools
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import polars.selectors as cs
import seaborn as sns
from tqdm import tqdm

out_dir = Path("out")
out_dir.mkdir(exist_ok=True, parents=True)

pl.Config.set_tbl_rows(100)

train_df = pl.read_csv("../data/raw/train.csv")
test_df = pl.read_csv("../data/raw/test.csv")
submit_df = pl.read_csv("../data/raw/sample_submission.csv")

train_df.head()

id,date,country,store,product,num_sold
i64,str,str,str,str,f64
0,"""2010-01-01""","""Canada""","""Discount Stickers""","""Holographic Goose""",null
1,"""2010-01-01""","""Canada""","""Discount Stickers""","""Kaggle""",973.0
2,"""2010-01-01""","""Canada""","""Discount Stickers""","""Kaggle Tiers""",906.0
3,"""2010-01-01""","""Canada""","""Discount Stickers""","""Kerneler""",423.0
4,"""2010-01-01""","""Canada""","""Discount Stickers""","""Kerneler Dark Mode""",491.0


In [2]:
train_df = train_df.drop("id").cast({"date": pl.Date})
test_df = test_df.drop("id").cast({"date": pl.Date})

train_df.describe()

statistic,date,country,store,product,num_sold
str,str,str,str,str,f64
"""count""","""230130""","""230130""","""230130""","""230130""",221259.0
"""null_count""","""0""","""0""","""0""","""0""",8871.0
"""mean""","""2013-07-02 00:00:00""",null,null,null,752.527382
"""std""",null,null,null,null,690.165445
"""min""","""2010-01-01""","""Canada""","""Discount Stickers""","""Holographic Goose""",5.0
"""25%""","""2011-10-02""",null,null,null,219.0
"""50%""","""2013-07-02""",null,null,null,605.0
"""75%""","""2015-04-02""",null,null,null,1114.0
"""max""","""2016-12-31""","""Singapore""","""Stickers for Less""","""Kerneler Dark Mode""",5939.0


In [3]:
(
    train_df.group_by("country", "store", "product")
    .agg(pl.col("num_sold").null_count())
    .filter(pl.col("num_sold") > 0)
)

country,store,product,num_sold
str,str,str,u32
"""Kenya""","""Discount Stickers""","""Holographic Goose""",2557
"""Canada""","""Discount Stickers""","""Holographic Goose""",2557
"""Canada""","""Discount Stickers""","""Kerneler""",1
"""Kenya""","""Discount Stickers""","""Kerneler Dark Mode""",1
"""Canada""","""Stickers for Less""","""Holographic Goose""",1308
"""Kenya""","""Discount Stickers""","""Kerneler""",63
"""Canada""","""Premium Sticker Mart""","""Holographic Goose""",380
"""Kenya""","""Premium Sticker Mart""","""Holographic Goose""",646
"""Kenya""","""Stickers for Less""","""Holographic Goose""",1358


In [4]:
countries = train_df["country"].unique().to_list()
stores = train_df["store"].unique().to_list()
products = train_df["product"].unique().to_list()

out_dir = Path("./out/num_sold_density")
shutil.rmtree(out_dir, ignore_errors=True)
out_dir.mkdir(exist_ok=True, parents=True)
for category in ["country", "store", "product"]:
    tmp = (
        train_df.drop_nulls().group_by("date", category).agg(pl.col("num_sold").mean())
    )
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.histplot(tmp, x="num_sold", hue=category, kde=True, alpha=0.2, ax=ax)
    ax.set_title(f"num_sold density by {category}")
    ax.set_xlabel("num_sold")
    ax.set_ylabel("density")
    fig.savefig(out_dir / f"{category}.png")
    fig.clear()
    plt.close(fig)


In [5]:
out_dir = Path("./out/num_sold_over_date")
shutil.rmtree(out_dir, ignore_errors=True)
out_dir.mkdir(exist_ok=True, parents=True)
for category in ["country", "store", "product"]:
    tmp = (
        train_df.drop_nulls().group_by("date", category).agg(pl.col("num_sold").mean())
    )
    fig, ax = plt.subplots(figsize=(20, 8))
    sns.lineplot(data=tmp, x="date", y="num_sold", hue=category, ax=ax)
    ax.set_title(f"num_sold over date by {category}")
    ax.set_xlabel("date")
    ax.set_ylabel("num_sold")
    fig.savefig(out_dir / f"{category}.png")
    fig.clear()
    plt.close(fig)